# Benchmark — Qdrant
## Vade Mecum EC134/2024

Pipeline completo de ingestão e busca com o backend **Qdrant**.

| Passo | Descrição |
|-------|-----------|
| 0 | Verificar conexão ao Qdrant |
| 1 | Extrair texto do PDF (PyMuPDF) |
| 2 | Chunking jurídico por artigo |
| 3 | Gerar embeddings (multilingual-e5-large, 1024 dims) |
| 4 | Indexar no Qdrant (HNSW + cosine) |
| 5 | Buscas de teste — latência por query |
| 6 | Estatísticas de storage |

### Pré-requisitos

```bash
# Suba o Qdrant
docker compose up qdrant -d
```

> Modelo `intfloat/multilingual-e5-large` (~2 GB) baixado automaticamente na 1ª execução.  
> `RECRIAR = True` → recria a collection; `False` → reutiliza dados já indexados.


In [1]:
import sys
import statistics
import time
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CAMINHO_PDF = ROOT / 'data' / 'Vade_mecum_EC134_2024.pdf'
RECRIAR = True  # False = reutiliza dados já indexados (pula ingestão)

QUERIES = [
    'princípios do tratamento de dados pessoais LGPD',
    'direitos fundamentais habeas corpus mandado de segurança',
    'rescisão do contrato de trabalho aviso prévio CLT',
    'direitos do consumidor código de defesa vício do produto',
    'imposto de renda pessoa física alíquota tributação',
    'usucapião direito de propriedade prazo posse',
    'crime de furto pena reclusão detenção código penal',
    'ação civil pública interesse difuso ministério público',
    'licitação contrato administrativo dispensa inexigibilidade',
    'criança adolescente ECA medida socioeducativa',
]

_t = {}  # acumula tempos para o resumo final
assert CAMINHO_PDF.exists(), f'PDF não encontrado: {CAMINHO_PDF}'
print(f'✔  {CAMINHO_PDF.name}  ({CAMINHO_PDF.stat().st_size / 1e6:.1f} MB)')

✔  Vade_mecum_EC134_2024.pdf  (24.4 MB)


## Passo 0 — Conectividade

Verifica se o Qdrant está rodando na porta 6333.

In [2]:
from ana.rag.indexador import IndexadorQdrant

idx = IndexadorQdrant()
if not idx.verificar_conexao():
    raise RuntimeError('Qdrant indisponível. Execute: docker compose up qdrant -d')
print('✔  Qdrant acessível')
print(f'   Collections: {idx.listar_colecoes()}')

✔  Qdrant acessível
   Collections: ['legislacao_brasileira']


## Passo 1 — Extração de texto do PDF

Usa **PyMuPDF** para extrair o texto bruto. O Vade Mecum tem ~4.4 M de caracteres.

In [3]:
from ana.rag.ingestao import extrair_texto_pdf

t0 = time.perf_counter()
texto = extrair_texto_pdf(CAMINHO_PDF)
t_ext = time.perf_counter() - t0
_t['extracao'] = t_ext

pdf_mb = CAMINHO_PDF.stat().st_size / 1e6
print(f'✔  {len(texto):,} caracteres extraídos  [{t_ext:.2f}s]')
print(f'   Velocidade: {pdf_mb / t_ext:.1f} MB/s')
print()
print('Primeiros 500 chars:')
print(texto[:500])

2026-03-03 07:12:36.330 | INFO     | ana.rag.ingestao:extrair_texto_pdf:270 - PDF extraído: Vade_mecum_EC134_2024.pdf (4393818 chars)


✔  4,393,818 caracteres extraídos  [2.49s]
   Velocidade: 9.8 MB/s

Primeiros 500 chars:



Brasília – DF
VADE 
MECUM

Mesa Diretora do Senado Federal
Biênio 2023–2024
Senador Rodrigo Pacheco
PRESIDENTE
Senador Veneziano Vital do Rêgo
PRIMEIRO-VICE-PRESIDENTE
Senador Rodrigo Cunha
SEGUNDO-VICE-PRESIDENTE
Senador Rogério Carvalho
PRIMEIRO-SECRETÁRIO
Senador Weverton
SEGUNDO-SECRETÁRIO
Senador Chico Rodrigues
TERCEIRO-SECRETÁRIO
Senador Styvenson Valentim
QUARTO-SECRETÁRIO
SUPLENTES DE SECRETÁRIO
1ª suplente: Senadora Mara Gabrilli
2ª suplente: Senadora Ivete da Silveira
3o suplente: 


## Passo 2 — Chunking jurídico por artigo

O texto é dividido em chunks usando o padrão `Art. N`, preservando o texto completo de cada dispositivo legal.

In [4]:
from ana.rag.ingestao import processar_documento
from ana.rag.modelos import TipoDocumento, VigenciaStatus

t0 = time.perf_counter()
chunks = processar_documento(
    texto=texto,
    fonte='Vade Mecum EC134/2024',
    tipo=TipoDocumento.LEI_FEDERAL,
    vigencia=VigenciaStatus.ATIVA,
)
t_chk = time.perf_counter() - t0
_t['chunking'] = t_chk

tamanhos = [len(c.texto) for c in chunks]
print(f'✔  {len(chunks):,} chunks gerados  [{t_chk:.2f}s]')
print(f'   média={statistics.mean(tamanhos):.0f}  min={min(tamanhos)}  max={max(tamanhos)}  mediana={statistics.median(tamanhos):.0f} chars')

2026-03-03 07:12:38.195 | INFO     | ana.rag.ingestao:chunkar_texto_juridico:228 - Chunking concluído: 7889 artigos extraídos de 'Vade Mecum EC134/2024'


✔  7,889 chunks gerados  [1.86s]
   média=520  min=17  max=21822  mediana=295 chars


In [5]:
# Amostra dos primeiros 10 chunks
pd.DataFrame([
    {'artigo': c.metadata.artigo, 'fonte': c.metadata.fonte, 'chars': len(c.texto), 'prévia': c.texto[:100] + '…'}
    for c in chunks[:10]
])

,artigo,fonte,chars,prévia
0,Art. 1,Vade Mecum EC134/2024,490,"Art. 1o A República Federativa do Brasil, for..."
1,Art. 2,Vade Mecum EC134/2024,110,"Art. 2o São Poderes da União, independentes e..."
2,Art. 3,Vade Mecum EC134/2024,392,Art. 3o Constituem objetivos fundamentais da...
3,Art. 4,Vade Mecum EC134/2024,704,Art. 4o A República Federativa do Brasil rege...
4,Art. 5,Vade Mecum EC134/2024,13627,"Art. 5o Todos são iguais perante a lei, sem d..."
5,Art. 6,Vade Mecum EC134/2024,548,"Art. 6o São direitos sociais a educação, a sa..."
6,Art. 7,Vade Mecum EC134/2024,4570,Art. 7o São direitos dos trabalhadores urbano...
7,Art. 8,Vade Mecum EC134/2024,1626,Art. 8o É livre a associação profissional ou ...
8,Art. 9,Vade Mecum EC134/2024,367,"Art. 9o É assegurado o direito de greve, comp..."
9,Art. 10,Vade Mecum EC134/2024,202,Art. 10. É assegurada a participação dos trab...


## Passo 3 — Geração de embeddings

Usa **`intfloat/multilingual-e5-large`** (1024 dims) via CUDA. Na 1ª execução faz o download do modelo (~2 GB).

In [6]:
from ana.rag.embeddings import GeradorEmbeddings
from ana.config_modelos import obter_modelos

cfg_emb = obter_modelos().ativo.embeddings
print(f'Modelo  : {cfg_emb.modelo}')
print(f'Dimensão: {cfg_emb.dimensao}')
print(f'Device  : {cfg_emb.dispositivo}')

gerador = GeradorEmbeddings()
textos = [c.texto for c in chunks]

t0 = time.perf_counter()
vecs = gerador.gerar_batch(textos)
t_emb = time.perf_counter() - t0
_t['embeddings'] = t_emb

for chunk, v in zip(chunks, vecs):
    chunk.embedding = v

n = len(chunks)
print(f'\n✔  {n:,} embeddings  [{t_emb:.2f}s]  ({n / t_emb:.0f} chunks/s)')
print(f'   Shape: {len(vecs)} × {len(vecs[0])} dims')

Modelo  : intfloat/multilingual-e5-large
Dimensão: 1024
Device  : cuda


/mnt/hd/Repos/attorney-normative-assistent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-03 07:12:59.946 | INFO     | ana.rag.embeddings:_carregar_modelo:73 - Carregando modelo de embeddings: intfloat/multilingual-e5-large (dispositivo=cuda)
2026-03-03 07:13:05.562 | INFO     | ana.rag.embeddings:_carregar_modelo:81 - Modelo carregado: dimensão=1024, batch_size=32
2026-03-03 07:13:05.566 | DEBUG    | ana.rag.embeddings:gerar_batch:124 - Gerando embeddings para 7889 textos...
Batches: 100%|██████████| 247/247 [01:10<00:00,  3.49it/s]



✔  7,889 embeddings  [98.40s]  (80 chunks/s)
   Shape: 7889 × 1024 dims


## Passo 4 — Indexação no Qdrant

Cria (ou recria) a collection `legislacao_brasileira` e insere os chunks via upsert em lotes de 100.

In [7]:
idx.criar_colecao_legislacao(recriar=RECRIAR)

t0 = time.perf_counter()
total = idx.indexar_chunks(chunks)
t_idx = time.perf_counter() - t0
_t['indexacao'] = t_idx

print(f'✔  {total:,} chunks indexados  [{t_idx:.2f}s]  ({total / t_idx:.0f} chunks/s)')

2026-03-03 07:14:16.675 | WARNING  | ana.rag.indexador:criar_colecao:99 - Recriando collection: legislacao_brasileira
2026-03-03 07:14:17.065 | INFO     | ana.rag.indexador:criar_colecao:128 - Collection criada: legislacao_brasileira (dims=1024, distância=COSINE)
2026-03-03 07:14:17.111 | DEBUG    | ana.rag.indexador:indexar_chunks:251 - Indexado lote 1: 100 chunks
2026-03-03 07:14:17.154 | DEBUG    | ana.rag.indexador:indexar_chunks:251 - Indexado lote 2: 100 chunks
2026-03-03 07:14:17.196 | DEBUG    | ana.rag.indexador:indexar_chunks:251 - Indexado lote 3: 100 chunks
2026-03-03 07:14:17.238 | DEBUG    | ana.rag.indexador:indexar_chunks:251 - Indexado lote 4: 100 chunks
2026-03-03 07:14:17.280 | DEBUG    | ana.rag.indexador:indexar_chunks:251 - Indexado lote 5: 100 chunks
2026-03-03 07:14:17.322 | DEBUG    | ana.rag.indexador:indexar_chunks:251 - Indexado lote 6: 100 chunks
2026-03-03 07:14:17.364 | DEBUG    | ana.rag.indexador:indexar_chunks:251 - Indexado lote 7: 100 chunks
2026-03-

✔  7,889 chunks indexados  [3.24s]  (2437 chunks/s)


## Passo 5 — Buscas de teste

10 queries jurídicas representativas. Mede apenas a latência da busca vetorial (o embedding da query não é contabilizado).

In [8]:
latencias = []
linhas = []

for query in QUERIES:
    vetor = gerador.gerar_query(query)
    t0 = time.perf_counter()
    res = idx.busca_semantica(vetor, limite=5)
    lat_ms = (time.perf_counter() - t0) * 1000
    latencias.append(lat_ms)

    melhor = res[0] if res else {}
    payload = melhor.get('payload', {})
    linhas.append({
        'query': query,
        'lat (ms)': round(lat_ms, 1),
        'artigo': payload.get('artigo', '—'),
        'score': round(melhor.get('score', 0), 3) if res else 0,
        'trecho': (payload.get('texto') or '')[:80] + '…',
    })

pd.DataFrame(linhas)

,query,lat (ms),artigo,score,trecho
0,princípios do tratamento de dados pessoais LGPD,5.4,Art. 49,0.839,Art. 49. As entidades que desenvolvam program...
1,direitos fundamentais habeas corpus mandado de...,2.7,Art. 647,0.873,Art. 647. Dar-se-á habeas corpus sempre que a...
2,rescisão do contrato de trabalho aviso prévio CLT,2.6,Art. 487,0.882,"Art. 487. Não havendo prazo estipulado, a par..."
3,direitos do consumidor código de defesa vício ...,2.8,Art. 18,0.876,Art. 18. Os fornecedores de produtos de consu...
4,imposto de renda pessoa física alíquota tribut...,2.6,Art. 127,0.869,"Art. 127. Em 2027 e 2028, o imposto previsto ..."
5,usucapião direito de propriedade prazo posse,2.6,Art. 1,0.894,Art. 1.379. O exercício incontestado e contín...
6,crime de furto pena reclusão detenção código p...,2.7,Art. 351,0.877,Art. 351. Promover ou facilitar a fuga de pes...
7,ação civil pública interesse difuso ministério...,2.3,Art. 77,0.870,Art. 77. A falta de intervenção do Ministério...
8,licitação contrato administrativo dispensa ine...,2.6,Art. 72,0.868,"Art. 72. O processo de contratação direta, qu..."
9,criança adolescente ECA medida socioeducativa,2.2,Art. 18-B,0.863,"Art. 18-B. Os pais, os integrantes da família..."


In [9]:
lat_s = sorted(latencias)
p95 = lat_s[int(len(lat_s) * 0.95)]
print('Latências (ms):')
print(f'  média : {statistics.mean(latencias):.1f}')
print(f'  p50   : {statistics.median(latencias):.1f}')
print(f'  p95   : {p95:.1f}')
print(f'  max   : {max(latencias):.1f}')
_t['latencias'] = latencias

Latências (ms):
  média : 2.9
  p50   : 2.6
  p95   : 5.4
  max   : 5.4


## Passo 6 — Estatísticas de storage

Consulta o Qdrant via REST API e cliente Python.

In [10]:
from ana.config import obter_configuracao
import httpx

cfg = obter_configuracao()
colecao = cfg.colecao_legislacao

info = idx.cliente.get_collection(colecao)
print(f'Pontos indexados   : {info.points_count:,}')
print(f'Segmentos          : {info.segments_count}')

url = f'http://{cfg.qdrant_host}:{cfg.qdrant_port}/collections/{colecao}'
result = httpx.get(url, timeout=5).json().get('result', {})
disk = result.get('disk_data_size')
ram  = result.get('ram_data_size')
if disk:
    print(f'Disco              : {disk / 1e6:.1f} MB')
if ram:
    print(f'RAM                : {ram / 1e6:.1f} MB')

est_mb = len(chunks) * 1024 * 4 / 1e6
print(f'Estimativa vetores : {est_mb:.1f} MB  (antes de quantização/compressão)')

Pontos indexados   : 7,889
Segmentos          : 4
Estimativa vetores : 32.3 MB  (antes de quantização/compressão)


## Resumo Final

In [11]:
lats  = _t['latencias']
lat_s = sorted(lats)
t_ext = _t['extracao']
t_chk = _t['chunking']
t_emb = _t['embeddings']
t_idx = _t['indexacao']
n     = len(chunks)
total_s = t_ext + t_chk + t_emb + t_idx
pdf_mb  = CAMINHO_PDF.stat().st_size / 1e6

resumo = pd.DataFrame([
    {'Etapa': 'Extração PDF',          'Tempo': f'{t_ext:.2f}s',   'Throughput': f'{pdf_mb / t_ext:.1f} MB/s'},
    {'Etapa': 'Chunking jurídico',     'Tempo': f'{t_chk:.2f}s',   'Throughput': f'{n / t_chk:.0f} chunks/s'},
    {'Etapa': 'Embeddings (e5-large)', 'Tempo': f'{t_emb:.2f}s',   'Throughput': f'{n / t_emb:.0f} chunks/s'},
    {'Etapa': 'Indexação Qdrant',      'Tempo': f'{t_idx:.2f}s',   'Throughput': f'{n / t_idx:.0f} chunks/s'},
    {'Etapa': 'TOTAL ingestão',        'Tempo': f'{total_s:.1f}s', 'Throughput': ''},
    {'Etapa': 'Busca — média',         'Tempo': '',                'Throughput': f'{statistics.mean(lats):.1f} ms'},
    {'Etapa': 'Busca — p50',           'Tempo': '',                'Throughput': f'{statistics.median(lats):.1f} ms'},
    {'Etapa': 'Busca — p95',           'Tempo': '',                'Throughput': f'{lat_s[int(len(lat_s) * 0.95)]:.1f} ms'},
    {'Etapa': 'Busca — max',           'Tempo': '',                'Throughput': f'{max(lats):.1f} ms'},
])
resumo.style.set_caption('Benchmark Qdrant — Vade Mecum EC134/2024')

,Etapa,Tempo,Throughput
0,Extração PDF,2.49s,9.8 MB/s
1,Chunking jurídico,1.86s,4239 chunks/s
2,Embeddings (e5-large),98.40s,80 chunks/s
3,Indexação Qdrant,3.24s,2437 chunks/s
4,TOTAL ingestão,106.0s,
5,Busca — média,,2.9 ms
6,Busca — p50,,2.6 ms
7,Busca — p95,,5.4 ms
8,Busca — max,,5.4 ms
